# HE-IFD -- Federated Fine-Tuning Experiments\n\n**Run All** (Runtime -> Run all). Nothing to configure. Each stage runs unattended and prints its results as CSV inline -- copy the block between the `===== BEGIN results.csv =====` markers (no download). Stages share one case (`ft_experiments`) and **resume** each other, so if Colab disconnects just Run All again.\n\n- **Setup**: clone, deps, prefetch datasets, stay online.\n- **Stage A (quick)**: 4 cells, first numbers in minutes.\n- **Stage B (headline)**: the full grid -- hard many-class text + a vision reference, across heterogeneity and 3 seeds, with and without the shared basin.\n- **Stage C (optional)**: fine-grained vision, scaling, lambda, trainable-unit and distillation ablations.

In [ ]:
# Cell 1 -- import os, sys, subprocess
REPO_DIR, REPO_URL = "/content/HE-IFD", "https://github.com/hkanpak21/HE-IFD.git"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git","clone","-q",REPO_URL,REPO_DIR], check=False)
os.chdir(REPO_DIR)
subprocess.run(["git","fetch","-q","origin","master"], check=False)
subprocess.run(["git","checkout","-q","origin/master","--","src","jobs","tests"], check=False)
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)

# Deps. PIN datasets<3.0 -- newer datasets removed the script loaders that
# Banking77 / TREC / DBpedia still ship ("Dataset scripts are no longer supported").
subprocess.run([sys.executable,"-m","pip","-q","install",
                "datasets<3.0","transformers","timm","ipywidgets"], check=False)
os.environ["HF_DATASETS_TRUST_REMOTE_CODE"] = "1"   # auto-accept those dataset scripts

import torch
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

# Vision: download directly via torchvision (guaranteed; no datasets lib involved).
import torchvision as tv
for ds in (tv.datasets.CIFAR100, tv.datasets.CIFAR10):
    ds("data", train=True, download=True); ds("data", train=False, download=True)
print("cifar100 / cifar10 ready under data/")

# Text datasets + models via the helper. Split so one dataset failing cannot
# abort the rest. (datasets<3 + trust_remote_code now load the scripts.)
for flags in (["--include-text019"], ["--include-ft02-text"]):
    print("prefetching:", flags)
    subprocess.run([sys.executable,"jobs/prefetch_login.py","--data-root","data",*flags], check=False)

os.environ.pop("HF_HUB_OFFLINE", None); os.environ.pop("TRANSFORMERS_OFFLINE", None)
print("setup done -- online; cifar + text datasets cached.")
print("NOTE: if a TEXT cell still says 'Dataset scripts...', do Runtime -> Restart and run all "
      "(datasets was already imported before the pin took effect).")


## Stage A -- quick smoke (first numbers)

In [ ]:
# Cell 2 -- STAGE A: quick smoke, ~4 cells, first numbers in minutes.
from src.notebook_runner import run_unattended
run_unattended(dict(
    backbones=["roberta_base_banking77", "vit_b32_cifar100"],
    Ns=[10], alphas=[0.05, 1.0],
    methods=["raw_union_K300"], seeds=[42], Ks=[300],
    trainable_units=["lora"], local_steps=["finetune"],
    case="ft_experiments",
))

## Stage B -- headline grid

In [ ]:
# Cell 3 -- STAGE B: headline grid (the experiments we need). ~120 cells; resumes past Stage A.
# Hard many-class text (Banking77/DBpedia/TREC/20NG) + CIFAR-100 vision reference,
# across heterogeneity alpha and 3 seeds, WITH the shared basin (raw_union) and WITHOUT it
# (no_phase0) so the basin's necessity is visible. LoRA + direct fine-tuning throughout.
from src.notebook_runner import run_unattended
run_unattended(dict(
    backbones=["roberta_base_banking77", "roberta_base_dbpedia",
               "roberta_base_trec", "roberta_base_20news", "vit_b32_cifar100"],
    Ns=[10], alphas=[0.05, 0.1, 0.3, 1.0],
    methods=["raw_union_K300", "no_phase0"], seeds=[42, 43, 44], Ks=[300],
    trainable_units=["lora"], local_steps=["finetune"],
    case="ft_experiments",
))

## Stage C -- optional extensions (uncomment a block)

In [ ]:
# Cell 4 -- STAGE C (OPTIONAL): uncomment a block to run it. Same case -> resumes.
from src.notebook_runner import run_unattended

# Fine-grained vision -- FIRST add "--include-ft02-fgvc" to PREFETCH_FLAGS in Setup and re-run Setup.
# run_unattended(dict(backbones=["vit_b32_fgvc_aircraft"], Ns=[10], alphas=[0.05,0.1,0.3,1.0],
#     methods=["raw_union_K300","no_phase0"], seeds=[42,43,44], Ks=[300],
#     trainable_units=["lora"], local_steps=["finetune"], case="ft_experiments"))

# Scaling in number of clients.
# run_unattended(dict(backbones=["roberta_base_banking77"], Ns=[5,20,50], alphas=[0.05,1.0],
#     methods=["raw_union_K300"], seeds=[42,43,44], Ks=[300],
#     trainable_units=["lora"], local_steps=["finetune"], case="ft_experiments"))

# Trainable-unit comparison: head (linear probe) vs LoRA vs last-N blocks.
# run_unattended(dict(backbones=["roberta_base_banking77"], Ns=[10], alphas=[0.05,1.0],
#     methods=["raw_union_K300"], seeds=[42,43,44], Ks=[300],
#     trainable_units=["head","lora","last_n"], local_steps=["finetune"], case="ft_experiments"))

# Lambda scaling-coefficient curve (eval-only drift regularizer, issue 026).
# run_unattended(dict(backbones=["roberta_base_banking77"], Ns=[10], alphas=[0.05,1.0],
#     methods=["raw_union_K300"], seeds=[42], Ks=[300], trainable_units=["lora"], local_steps=["finetune"],
#     lambda_scales=[0,0.25,0.5,0.75,1.0,1.25,1.5,1.75,2.0], case="ft_experiments"))

# Distillation ablation (teacher-based local step instead of direct fine-tuning).
# run_unattended(dict(backbones=["roberta_base_banking77"], Ns=[10], alphas=[0.05,1.0],
#     methods=["raw_union_K300"], seeds=[42,43,44], Ks=[300],
#     trainable_units=["lora"], local_steps=["distill"], case="ft_experiments"))